# Neuron Flow 0.4.0.dev0

A supervised document-validation workflow. This notebook demonstrates CSV intake, persistent batch IDs, report inspection, recovery, verification, backup and restore.

Supports 46 currencies with currency-specific minor units and separate certified totals. Compatible CSV workspaces retain their original source, contracts, creation version and reports. Actual Colab acceptance remains `not_run` until fresh-runtime upload, execution, recovery, download receipt and restore are evidenced.

For PDF, OCR, Excel and cloud connectors, see `docs/INTEGRATIONS.md` and install the selected optional extras.


## 1. Upload and verify the project ZIP

Select exactly **neuron_flow_v0.4.0.dev0.zip**. The cell validates ZIP paths and sizes and checks SHA-256 for every file before executing project code. The core CSV workflow requires Python 3.10+ and no additional dependencies. Run once per fresh runtime.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from contextlib import closing
import ast
import csv
import hashlib
import io
import json
import os
import platform
import shutil
import sqlite3
import stat
import subprocess
import sys
import tempfile
import zipfile

if sys.version_info < (3, 10):
    raise RuntimeError('Python 3.10 or newer is required')

PROJECT = Path('/content/neuron_flow_project/neuron_flow')
WORKSPACE = Path('/content/neuron_flow_workspace')
SESSION_PATH = Path('/content/neuron_flow_session.json')
UPLOAD_DIRECTORY = Path('/content/neuron_flow_uploads')
OBSERVATIONS = {'runtime': 'Google Colab', 'python': sys.version, 'platform': platform.platform(),
                'started_at_utc': datetime.now(timezone.utc).isoformat(),
                'download_received_confirmed': False,
                'runtime_marker_names': [name for name in ('COLAB_RELEASE_TAG', 'COLAB_BACKEND_VERSION', 'COLAB_JUPYTER_TRANSPORT') if name in os.environ]}

def safe_unpack(data, destination, allowed_roots):
    if len(data) > 64 * 1024 * 1024:
        raise ValueError('ZIP must not exceed 64 MiB')
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        entries = archive.infolist()
        if len(entries) > 5000 or sum(item.file_size for item in entries) > 256 * 1024 * 1024:
            raise ValueError('ZIP entry count or expanded size exceeds the limit')
        seen = set()
        for item in entries:
            relative = PurePosixPath(item.filename)
            mode = (item.external_attr >> 16) & 0xffff
            if (not relative.parts or relative.is_absolute() or '..' in relative.parts
                    or '\\' in item.filename or any(':' in part for part in relative.parts)
                    or relative.parts[0] not in allowed_roots
                    or stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR)
                    or item.flag_bits & 1 or item.file_size > 64 * 1024 * 1024):
                raise ValueError('ZIP contains an unsupported path or file type')
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()) or target in seen:
                raise ValueError('ZIP contains duplicate paths or paths outside the destination')
            seen.add(target)
        archive.extractall(destination)

def verify_sums(root, manifest_name='SHA256SUMS.txt'):
    root = Path(root)
    checked = set()
    for line in (root / manifest_name).read_text(encoding='utf-8').splitlines():
        digest, separator, name = line.partition('  ')
        relative = PurePosixPath(name)
        if (not separator or len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest)
                or not relative.parts or relative.is_absolute() or '..' in relative.parts
                or '\\' in name or name == manifest_name or name in checked):
            raise ValueError('Invalid SHA-256 manifest format')
        path = root / name
        if (not path.resolve().is_relative_to(root.resolve()) or not path.is_file()
                or path.is_symlink() or hashlib.sha256(path.read_bytes()).hexdigest() != digest):
            raise ValueError('SHA-256 mismatch: ' + name)
        checked.add(name)
    actual = {p.relative_to(root).as_posix() for p in root.rglob('*')
              if p.is_file() and p.relative_to(root).as_posix() != manifest_name}
    if not checked or checked != actual:
        raise ValueError('Manifest does not cover every file')
    return len(checked)

if PROJECT.exists():
    raise RuntimeError('A project is already installed; continue with the next cell or use a fresh runtime')
uploaded = files.upload()
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
    raise ValueError('Select exactly one project ZIP')
safe_unpack(next(iter(uploaded.values())), PROJECT.parent, {'neuron_flow'})
verified_files = verify_sums(PROJECT)
required = ['neuron_flow/__main__.py', 'neuron_flow/batch.py', 'neuron_flow/reporting.py', 'neuron_flow/currencies.py',
            'tools/verify_v020.py', 'acceptance/fixture_lock.json']
if not all((PROJECT / name).is_file() for name in required):
    raise ValueError('Incomplete project package')
version = subprocess.check_output([sys.executable, '-c',
    'from neuron_flow import __version__; print(__version__)'], cwd=PROJECT, text=True).strip()
if version != '0.4.0.dev0':
    raise ValueError('The ZIP must contain version 0.4.0.dev0')
LEGACY_020_MANIFEST_SHA = '0af9c310e75717386bcf2cd4cc00a5eeb92a8c5de5ee819bd030681d2ce2f41a'
PACKAGE_MANIFEST_SHA = hashlib.sha256((PROJECT / 'SHA256SUMS.txt').read_bytes()).hexdigest()
OBSERVATIONS.update(project_version=version, project_files_verified=verified_files,
                    package_manifest_sha256=PACKAGE_MANIFEST_SHA)

def run_cli(*arguments):
    completed = subprocess.run([sys.executable, '-m', 'neuron_flow', *map(str, arguments)],
                               cwd=PROJECT, capture_output=True, text=True, timeout=300)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode not in (0, 1):
        print(completed.stdout)
        completed.check_returncode()
    result = json.loads(completed.stdout)
    print('exit code:', completed.returncode, 'status:', result.get('batch_status', result.get('status')))
    if 'batches' in result:
        print('discovered batches:', [(item['batch_id'], item['status']) for item in result['batches']])
    else:
        print('document counts:', result.get('counts', {}), 'currency:', result.get('currency'))
        print('certified totals:', result.get('certified_totals', result.get('certified_total')))
    if completed.returncode == 1:
        print('ERROR or incomplete delivery: inspect document results and receipts')
    return result

print('Ready:', PROJECT, '| Python', platform.python_version(), '| SHA-256 verified:', verified_files, 'files')


## 2. Upload one CSV and process it

Select one UTF-8 purchase CSV. The cell persists the batch ID before starting processing. When restoring a backup, skip this cell and continue at step 4.


In [ ]:
uploaded_csv = files.upload()
if len(uploaded_csv) != 1 or not next(iter(uploaded_csv)).lower().endswith('.csv'):
    raise ValueError('Select exactly one CSV')
csv_bytes = next(iter(uploaded_csv.values()))
if len(csv_bytes) > 16 * 1024 * 1024:
    raise ValueError('CSV must not exceed 16 MiB')
upload_directory = UPLOAD_DIRECTORY
upload_directory.mkdir(exist_ok=True)
source_path = upload_directory / 'input.csv'
source_path.write_bytes(csv_bytes)
SUBMISSION = run_cli('csv', source_path, '--submit-only', '--workspace', WORKSPACE)
BATCH_ID = SUBMISSION['batch_id']
SESSION_PATH.write_text(json.dumps({'batch_id': BATCH_ID, 'workspace': str(WORKSPACE),
    'project_version': version, 'package_manifest_sha256': PACKAGE_MANIFEST_SHA},
    ensure_ascii=False, indent=2), encoding='utf-8')
OBSERVATIONS.update(csv_uploaded=True, csv_sha256=hashlib.sha256(csv_bytes).hexdigest(),
                    batch_id=BATCH_ID, batch_persisted_before_execution=True)
print('Batch ID saved before processing:', BATCH_ID)
RESULT = run_cli('batch-resume', BATCH_ID, '--workspace', WORKSPACE)
OBSERVATIONS.update(cli_counts=RESULT.get('counts'), cli_executed=True)
print('ungrouped rows:', RESULT.get('ungrouped_rows'))
print(json.dumps(RESULT.get('receipt', {}), ensure_ascii=False, indent=2))


## 3. Inspect reports

Review document counts, currency-specific certified totals, ungrouped rows, original source records and error details. Only VALID documents contribute to certified totals. An exit code of zero does not mean every document is valid.


In [ ]:
session = json.loads(SESSION_PATH.read_text(encoding='utf-8'))
BATCH_ID = session['batch_id']
STATUS = run_cli('batch-status', BATCH_ID, '--workspace', WORKSPACE)
artifact_info = (STATUS.get('receipt') or {}).get('artifacts', {})
if not artifact_info:
    raise RuntimeError('Reports are incomplete; inspect status and use the resume cell')
summary_path = Path(artifact_info['summary.csv']['path'])
with summary_path.open(encoding='utf-8', newline='') as handle:
    preview = list(csv.DictReader(handle))[:10]
print(json.dumps(preview, ensure_ascii=False, indent=2))
details = json.loads(Path(artifact_info['details.json']['path']).read_bytes())
print('batch metadata:', {key: details.get(key) for key in
    ('counts', 'currency', 'certified_total', 'certified_totals', 'source_sha256')})
print('ungrouped_rows:', len(details['ungrouped_rows']))
OBSERVATIONS['report_previewed'] = True


## 4. Resume the saved batch

Use the original workspace. SQLite retains the CSV; no new upload is needed. Completed steps and verified artifacts are reused. If session metadata is missing, discover stored batches and select an explicit ID when several exist. After restoring a backup, run this cell.


In [ ]:
if not SESSION_PATH.is_file():
    found = run_cli('batch-list', '--workspace', WORKSPACE)['batches']
    if not found:
        raise RuntimeError('No batches found in this workspace')
    identifiers = [item['batch_id'] for item in found]
    chosen = identifiers[0] if len(identifiers) == 1 else input('Enter the batch ID to resume: ').strip()
    if chosen not in identifiers:
        raise ValueError('Batch ID is not in the discovered list')
    SESSION_PATH.write_text(json.dumps({'batch_id': chosen, 'workspace': str(WORKSPACE),
        'project_version': version, 'package_manifest_sha256': PACKAGE_MANIFEST_SHA},
        ensure_ascii=False, indent=2), encoding='utf-8')
    OBSERVATIONS['batch_discovery_used'] = True
session = json.loads(SESSION_PATH.read_text(encoding='utf-8'))
if session['workspace'] != str(WORKSPACE) or not (WORKSPACE / 'state.sqlite3').is_file():
    raise RuntimeError('Original workspace not found; restore its backup first')
BATCH_ID = session['batch_id']
RESULT = run_cli('batch-resume', BATCH_ID, '--workspace', WORKSPACE)
OBSERVATIONS.update(resume_executed=True, resume_counts=RESULT.get('counts'), batch_id=BATCH_ID)
print(json.dumps(RESULT.get('receipt', {}), ensure_ascii=False, indent=2))


## 5. Verify the implementation

Run the separate 38-case baseline, current regression suite and locked 100-document reference. The verifier records the actual test count, failures, errors and skips. Local tests and notebook syntax checks do not establish actual Colab acceptance.


In [ ]:
checked = subprocess.run([sys.executable, 'tools/verify_current.py'], cwd=PROJECT,
                         capture_output=True, text=True, timeout=300)
print(checked.stdout)
if checked.stderr:
    print(checked.stderr)
checked.check_returncode()
verification = json.loads((PROJECT / 'verification/v0_3_0/test_results.json').read_text(encoding='utf-8'))
print(json.dumps({key: verification[key] for key in
    ('version', 'baseline_38_passed', 'new_test_cases', 'failures', 'errors', 'skipped',
     'local_successful', 'colab', 'colab_acceptance_complete', 'release_ready', 'remaining_acceptance')}, ensure_ascii=False, indent=2))
OBSERVATIONS.update(verifier_executed=True, verifier_local_successful=verification['local_successful'],
                    observed_colab=verification['colab'])


## 6. Download the workspace and evidence

Wait for previous commands to finish. The cell takes a consistent SQLite backup and includes reports, intake bundles and session metadata. Download and retain the backup before ending the temporary Colab session. Restore requires the same absolute workspace path. Confirm actual receipt of the downloaded file separately.


In [ ]:
if not (WORKSPACE / 'state.sqlite3').is_file() or not SESSION_PATH.is_file():
    raise RuntimeError('No workspace or session is available to back up')
OBSERVATIONS['backup_prepared_at_utc'] = datetime.now(timezone.utc).isoformat()
observation_path = PROJECT / 'verification/v0_3_0/colab_notebook_session.json'
observation_path.parent.mkdir(parents=True, exist_ok=True)
observation_path.write_text(json.dumps(OBSERVATIONS, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
stage = Path(tempfile.mkdtemp(prefix='neuron_flow_backup_', dir='/content'))
shutil.copytree(WORKSPACE, stage / 'workspace', ignore=shutil.ignore_patterns(
    'state.sqlite3', 'state.sqlite3-wal', 'state.sqlite3-shm', '*.tmp'))
with closing(sqlite3.connect('file:' + str(WORKSPACE / 'state.sqlite3') + '?mode=ro', uri=True)) as original_db:
    with closing(sqlite3.connect(stage / 'workspace/state.sqlite3')) as backup_db:
        original_db.backup(backup_db)
shutil.copytree(PROJECT / 'verification', stage / 'verification')
shutil.copy2(SESSION_PATH, stage / 'session.json')
manifest = []
for path in sorted(stage.rglob('*')):
    if path.is_symlink():
        raise ValueError('The workspace contains a symlink; inspect it before backup')
    if path.is_file():
        manifest.append(hashlib.sha256(path.read_bytes()).hexdigest() + '  ' + path.relative_to(stage).as_posix())
(stage / 'backup_SHA256SUMS.txt').write_text('\n'.join(manifest) + '\n', encoding='utf-8')
verify_sums(stage, 'backup_SHA256SUMS.txt')
backup_path = Path('/content/neuron_flow_workspace_backup.zip')
with zipfile.ZipFile(backup_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(stage.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(stage).as_posix())
print('backup ZIP SHA-256:', hashlib.sha256(backup_path.read_bytes()).hexdigest())
print('Backup created:', backup_path, '| bytes:', backup_path.stat().st_size)
files.download(str(backup_path))


## Optional: restore a backup in a fresh runtime

Install the current project ZIP first, then select one backup ZIP and continue at step 4. The destination must not already exist. The cell accepts the pinned original 0.2.0 package backup or a backup made with the same current ZIP manifest. Other runtime-compatible workspaces do not automatically satisfy this package-identity check.


In [ ]:
if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
    raise RuntimeError('The workspace already exists; refusing to overwrite it')
restored_upload = files.upload()
if len(restored_upload) != 1 or not next(iter(restored_upload)).lower().endswith('.zip'):
    raise ValueError('Select exactly one backup ZIP')
restore_stage = Path(tempfile.mkdtemp(prefix='neuron_flow_restore_', dir='/content'))
safe_unpack(next(iter(restored_upload.values())), restore_stage,
            {'workspace', 'verification', 'session.json', 'backup_SHA256SUMS.txt'})
verify_sums(restore_stage, 'backup_SHA256SUMS.txt')
restored_session = json.loads((restore_stage / 'session.json').read_text(encoding='utf-8'))
backup_version = restored_session['project_version']
expected_manifest = {'0.2.0': LEGACY_020_MANIFEST_SHA, version: PACKAGE_MANIFEST_SHA}.get(backup_version)
if (restored_session['workspace'] != str(WORKSPACE) or expected_manifest is None
        or restored_session['package_manifest_sha256'] != expected_manifest):
    raise ValueError('Backup requires its original path and the supported 0.2.0 ZIP or the same current project ZIP')
if not (restore_stage / 'workspace/state.sqlite3').is_file():
    raise ValueError('Backup contains no SQLite workspace')
shutil.copytree(restore_stage / 'workspace', WORKSPACE, dirs_exist_ok=True)
shutil.copytree(restore_stage / 'verification', PROJECT / 'verification', dirs_exist_ok=True)
restored_session.update(restored_from_version=backup_version, project_version=version,
                        package_manifest_sha256=PACKAGE_MANIFEST_SHA)
SESSION_PATH.write_text(json.dumps(restored_session, ensure_ascii=False, indent=2), encoding='utf-8')
BATCH_ID = restored_session['batch_id']
OBSERVATIONS.update(backup_restored=True, batch_id=BATCH_ID)
print('Workspace restored:', WORKSPACE, '| batch_id:', BATCH_ID)
print('Continue at step 4 without uploading the CSV again')
